# 05 - Machine Learning Price Prediction

This notebook builds machine learning models to predict Airbnb listing prices in Amsterdam.

The goal is to use cleaned listing, host, property, review, availability, and neighbourhood features to estimate listing-level price.

This is a supervised regression problem because the target variable, price_numeric, is continuous.

Important: estimated_revenue_proxy is excluded from model features because it is calculated using price and would create target leakage.

In [1]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("Base libraries imported successfully.")

Base libraries imported successfully.


In [ ]:
%pip install scikit-learn

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Machine learning libraries imported successfully.")

Machine learning libraries imported successfully.


In [3]:
PROJECT_ROOT = Path("..")

WAREHOUSE_PATH = PROJECT_ROOT / "warehouse"
DUCKDB_DATABASE_PATH = WAREHOUSE_PATH / "airbnb_market.duckdb"

ML_REPORTS_PATH = PROJECT_ROOT / "reports" / "machine_learning"
ML_REPORTS_PATH.mkdir(parents=True, exist_ok=True)

ML_FIGURES_PATH = PROJECT_ROOT / "reports" / "figures" / "machine_learning"
ML_FIGURES_PATH.mkdir(parents=True, exist_ok=True)

print("DuckDB database exists:", DUCKDB_DATABASE_PATH.exists())
print("ML reports path:", ML_REPORTS_PATH)
print("ML figures path:", ML_FIGURES_PATH)

DuckDB database exists: True
ML reports path: ..\reports\machine_learning
ML figures path: ..\reports\figures\machine_learning


In [4]:
conn = duckdb.connect(
    database=str(DUCKDB_DATABASE_PATH),
    read_only=True
)

listing_master_df = conn.execute("""
SELECT *
FROM listing_master_final
""").fetchdf()

print("listing_master_df shape:", listing_master_df.shape)
listing_master_df.head()

listing_master_df shape: (10465, 101)


,listing_id,listing_name,host_id,host_profile_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,...,neighbourhood_min_price,neighbourhood_max_price,neighbourhood_avg_availability_rate,neighbourhood_avg_occupancy_proxy,neighbourhood_avg_review_score,neighbourhood_total_reviews,neighbourhood_avg_reviews_per_listing,neighbourhood_total_revenue_proxy,neighbourhood_avg_revenue_proxy,neighbourhood_dominant_room_type
0,28871,Comfortable double room,124245,1462510208428038912,Edwin,<NA>,Centrum-West,52.367750,4.89092,Private room,...,39.0,11412.0,0.34,0.66,4.82,116784,95.96,65092219.0,74221.46,Entire home/apt
1,29051,Comfortable single / double room,124245,1462510208428038912,Edwin,<NA>,Centrum-Oost,52.365840,4.89111,Private room,...,39.0,2477.0,0.36,0.64,4.81,85025,91.13,44237061.0,65246.40,Entire home/apt
2,44129,Luxury design with canal view,187728,1462512125293144576,Tanya,<NA>,Centrum-West,52.382110,4.88630,Entire home/apt,...,39.0,11412.0,0.34,0.66,4.82,116784,95.96,65092219.0,74221.46,Entire home/apt
3,44391,Quiet 2-bedroom Amsterdam city centre apartment,194779,1462512336903650304,Jan,<NA>,Centrum-Oost,52.371680,4.91471,Entire home/apt,...,39.0,2477.0,0.36,0.64,4.81,85025,91.13,44237061.0,65246.40,Entire home/apt
4,48373,Cozy family home in Amsterdam South,220434,1462513002935125760,Vesna & Misha,<NA>,Buitenveldert - Zuidas,52.327808,4.87680,Entire home/apt,...,40.0,1461.0,0.38,0.62,4.78,4212,35.39,4754582.0,54650.37,Entire home/apt


## 1. Machine Learning Dataset Preparation

This section prepares the dataset for price prediction.

Only listings with valid positive price values are used for supervised learning.  
Rows with missing or invalid price are excluded because the model needs a known target value during training.

In [5]:
ml_df = listing_master_df[
    listing_master_df["price_numeric"].notna() &
    (listing_master_df["price_numeric"] > 0)
].copy()

print("Total listings:", len(listing_master_df))
print("Listings available for ML price prediction:", len(ml_df))
print("Excluded rows with missing/invalid price:", len(listing_master_df) - len(ml_df))

Total listings: 10465
Listings available for ML price prediction: 6471
Excluded rows with missing/invalid price: 3994


In [6]:
target_column = "price_numeric"

y = ml_df[target_column].copy()

print("Target variable:", target_column)
print("Target row count:", len(y))
print("Target missing values:", y.isna().sum())

Target variable: price_numeric
Target row count: 6471
Target missing values: 0


In [7]:
candidate_numeric_features = [
    "minimum_nights",
    "number_of_reviews",
    "reviews_per_month",
    "calculated_host_listings_count",
    "availability_365",
    "availability_rate",
    "occupancy_proxy",
    "weekend_availability_rate",
    "weekday_availability_rate",
    "accommodates",
    "bathrooms",
    "bedrooms",
    "beds",
    "amenities_count",
    "review_scores_rating",
    "review_scores_cleanliness",
    "review_scores_location",
    "review_scores_value",
    "detailed_review_count",
    "unique_reviewer_count",
    "detailed_reviews_last_365d",
    "avg_reviews_per_year",
    "neighbourhood_median_price",
    "neighbourhood_avg_occupancy_proxy"
]

candidate_categorical_features = [
    "room_type",
    "property_type",
    "host_is_superhost",
    "host_identity_verified",
    "instant_bookable",
    "host_portfolio_segment",
    "availability_segment",
    "neighbourhood"
]

numeric_features = [
    column for column in candidate_numeric_features
    if column in ml_df.columns
]

categorical_features = [
    column for column in candidate_categorical_features
    if column in ml_df.columns
]

print("Numeric features:")
for column in numeric_features:
    print("-", column)

print("\nCategorical features:")
for column in categorical_features:
    print("-", column)

print("\nTotal numeric features:", len(numeric_features))
print("Total categorical features:", len(categorical_features))

Numeric features:
- minimum_nights
- number_of_reviews
- reviews_per_month
- calculated_host_listings_count
- availability_365
- availability_rate
- occupancy_proxy
- weekend_availability_rate
- weekday_availability_rate
- accommodates
- bathrooms
- bedrooms
- beds
- amenities_count
- review_scores_rating
- review_scores_cleanliness
- review_scores_location
- review_scores_value
- detailed_review_count
- unique_reviewer_count
- detailed_reviews_last_365d
- avg_reviews_per_year
- neighbourhood_median_price
- neighbourhood_avg_occupancy_proxy

Categorical features:
- room_type
- property_type
- host_is_superhost
- host_identity_verified
- instant_bookable
- host_portfolio_segment
- availability_segment
- neighbourhood

Total numeric features: 24
Total categorical features: 8


In [8]:
leakage_columns = [
    "price_numeric",
    "estimated_revenue_proxy",
    "unavailable_days_numeric"
]

selected_features = numeric_features + categorical_features

leakage_check = [
    column for column in leakage_columns
    if column in selected_features
]

print("Leakage columns found in selected features:", leakage_check)

Leakage columns found in selected features: []


In [9]:
X = ml_df[selected_features].copy()

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

X.head()

Feature matrix shape: (6471, 32)
Target shape: (6471,)


,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365,availability_rate,occupancy_proxy,weekend_availability_rate,weekday_availability_rate,accommodates,...,neighbourhood_median_price,neighbourhood_avg_occupancy_proxy,room_type,property_type,host_is_superhost,host_identity_verified,instant_bookable,host_portfolio_segment,availability_segment,neighbourhood
0,1.0,799,4.15,2.0,11,0.0301,0.9699,0.0481,0.0230,2.0,...,290.0,0.66,Private room,Private room in rental unit,True,True,<NA>,Small portfolio host,Low availability,Centrum-West
2,3.0,186,0.96,4.0,3,0.0082,0.9918,0.0096,0.0077,2.0,...,290.0,0.66,Entire home/apt,Entire rental unit,True,True,<NA>,Small portfolio host,Low availability,Centrum-West
5,2.0,656,3.45,1.0,278,0.7616,0.2384,0.7500,0.7663,3.0,...,290.0,0.66,Entire home/apt,Entire guest suite,True,True,<NA>,Single-listing host,Very high availability,Centrum-West
6,1.0,187,0.98,1.0,252,0.6904,0.3096,0.6731,0.6973,4.0,...,282.5,0.64,Entire home/apt,Entire condo,False,True,<NA>,Single-listing host,High availability,Centrum-Oost
7,1.0,631,3.35,1.0,242,0.6630,0.3370,0.6731,0.6590,2.0,...,290.0,0.66,Entire home/apt,Entire condo,True,True,<NA>,Single-listing host,High availability,Centrum-West


In [10]:
missing_feature_summary = (
    X.isna()
    .sum()
    .reset_index()
)

missing_feature_summary.columns = ["feature", "missing_count"]
missing_feature_summary["missing_percentage"] = (
    missing_feature_summary["missing_count"] / len(X) * 100
).round(2)

missing_feature_summary = missing_feature_summary.sort_values(
    "missing_percentage",
    ascending=False
)

missing_feature_summary

,feature,missing_count,missing_percentage
28,instant_bookable,6471,100.00
10,bathrooms,1160,17.93
11,bedrooms,1007,15.56
12,beds,861,13.31
15,review_scores_cleanliness,731,11.30
17,review_scores_value,731,11.30
16,review_scores_location,731,11.30
14,review_scores_rating,731,11.30
21,avg_reviews_per_year,649,10.03
2,reviews_per_month,647,10.00


In [11]:
missing_feature_summary_output_path = ML_REPORTS_PATH / "ml_feature_missing_summary.csv"

missing_feature_summary.to_csv(missing_feature_summary_output_path, index=False)

print(f"ML feature missing summary saved to: {missing_feature_summary_output_path}")

ML feature missing summary saved to: ..\reports\machine_learning\ml_feature_missing_summary.csv


In [14]:
fully_missing_features = missing_feature_summary[
    missing_feature_summary["missing_percentage"] == 100
]["feature"].tolist()

print("Fully missing features to remove:")
print(fully_missing_features)

selected_features_clean = [
    feature for feature in selected_features
    if feature not in fully_missing_features
]

numeric_features_clean = [
    feature for feature in numeric_features
    if feature in selected_features_clean
]

categorical_features_clean = [
    feature for feature in categorical_features
    if feature in selected_features_clean
]

X = ml_df[selected_features_clean].copy()

print("Original selected feature count:", len(selected_features))
print("Clean selected feature count:", len(selected_features_clean))
print("Clean numeric feature count:", len(numeric_features_clean))
print("Clean categorical feature count:", len(categorical_features_clean))
print("Updated feature matrix shape:", X.shape)

Fully missing features to remove:
['instant_bookable']
Original selected feature count: 32
Clean selected feature count: 31
Clean numeric feature count: 24
Clean categorical feature count: 7
Updated feature matrix shape: (6471, 31)


In [17]:
ml_dataset_overview = pd.DataFrame([
    {"metric": "total_source_rows", "value": len(listing_master_df)},
    {"metric": "ml_training_rows", "value": len(ml_df)},
    {"metric": "excluded_missing_or_invalid_price_rows", "value": len(listing_master_df) - len(ml_df)},
    {"metric": "numeric_feature_count", "value": len(numeric_features_clean)},
    {"metric": "categorical_feature_count", "value": len(categorical_features_clean)},
    {"metric": "total_feature_count", "value": len(selected_features_clean)},
    {"metric": "removed_fully_missing_feature_count", "value": len(fully_missing_features)}
])

ml_dataset_overview

,metric,value
0,total_source_rows,10465
1,ml_training_rows,6471
2,excluded_missing_or_invalid_price_rows,3994
3,numeric_feature_count,24
4,categorical_feature_count,7
5,total_feature_count,31
6,removed_fully_missing_feature_count,1


In [18]:
ml_dataset_overview_output_path = ML_REPORTS_PATH / "ml_dataset_overview.csv"

ml_dataset_overview.to_csv(ml_dataset_overview_output_path, index=False)

print(f"ML dataset overview saved to: {ml_dataset_overview_output_path}")

ML dataset overview saved to: ..\reports\machine_learning\ml_dataset_overview.csv


### Feature Selection Note

The `instant_bookable` feature was removed from the ML feature set because it was fully missing in the valid-price training dataset.

Fully missing features do not provide predictive value and may create preprocessing issues.  
Other partially missing numeric and categorical features are retained because they can be handled using imputation inside the machine learning pipeline.